# Spark-X2.6-4B-Loop72-LSA (`loop72-dlsa` Branch) — Official Benchmark & Modern 2026 Long-Context Evaluation Suite

This notebook evaluates **strictly the two core enhancements on the `loop72-dlsa` branch**:
1. **72-Visit Looped Transformer (`Loop72`)**:
   - **Prelude (`L0..L3`)** $\rightarrow$ **Pass 1 (`L4..L27`)** $\rightarrow$ **Pass 2 (`L4..L27` + `TwoStreamMHC`)** $\rightarrow$ **Pass 3 (`L12..L23` + `TwoStreamMHC`)** $\rightarrow$ **Coda (`L28..L35`)** = **`72` total layer visits** (`54` local sliding-window visits + `18` global full-attention visits).
   - **100% Frozen Backbone & Native `GELU` Preserved**: `preserve_native_activation=True`, `construct_engram=False` (**`0` Engram parameters allocated**), and last-position generation logit head (`lm_head(norm(h[:, -1:, :]))`).
2. **Per-Visit Paged Latent Sparse Attention (`D-LSA`)**:
   - **`VisitAwareDLSAIndexBank`**: `18` independent `FaithfulDLSAIndexer` modules (`1,769,472` total selector parameters across all 18 global visits: `L3` $\times 1$, `L7` $\times 2$, `L11` $\times 2$, `L15` $\times 3$, `L19` $\times 3$, `L23` $\times 3$, `L27` $\times 2$, `L31` $\times 1$, `L35` $\times 1$) operating on the **192-D RoPE-free key slice** (`partial_rotary_factor=0.25`).
   - **`Loop72HybridPagedCache`**: Exact `512`-token ring buffers on all `54` local visits (`108 MiB` total regardless of sequence length) + tiered `M1/M2` `64`-token KV pages (`bf16`, `int8`, `int4`, `mixed_rotary_bf16`) with flat or hierarchical two-level tree routing (`search_mode in {"flat", "tree"}`).

---

## Official `Spark X2.6-4B` Benchmark Reference Table

| Category | Benchmark | Hugging Face Dataset / Source | **Spark X2.6-4B** | Spark X2.5-4B | Qwen3.5 9B | Qwen3.5 4B | Gemma4 12B | Gemma4 E4B |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Overall** | **Average** | Official 26-Benchmark Macro Average | **63.3** | 52.1 | 49.1 | 43.2 | 47.0 | 29.7 |
| **Long Context** | **AA-LCRv1.1** | `ArtificialAnalysis/AA-LCR` | **67.3** | 64.7 | 70.0* | 63.0* | 63.7* | 32.0* |
| **Long Context** | **LongBench v2** | `zai-org/LongBench-v2` | **53.3** | 48.7 | 55.2* | 50.0* | 45.3 | 27.8 |
| **Long Context** | **Graphwalks bfs 256k f1** | `openai/graphwalks` | **92.2** | 47.1 | 88.4 | 79.0 | 68.6 | 54.8 |
| **Long Context** | **Graphwalks bfs 1m f1** | `openai/graphwalks` | **34.2** | 2.5 | 25.3 | 15.8 | - | - |
| **Long Context** | **Openai_MRCR v2 256k-512k** | `openai/mrcr` | **65.6** | 9.7 | 58.6 | 48.3 | - | - |
| **Long Context** | **Openai_MRCR v2 512k-1M** | `openai/mrcr` | **26.0** | 2.9 | 34.3 | 12.2 | - | - |
| **Math** | **Gaokao 2026** (out of 150) | `XHToken/gaokao_math_2026` | **134.9** | 133.4 | 135.5 | 130.3 | 130.6 | 102.4 |
| **Math** | **AIME 2026** | `MathArena/aime_2026` | **91.2** | 90.7 | 88.2 | 83.0 | 82.1* | 42.5* |
| **Math** | **HMMT Feb 26** | `MathArena/hmmt_feb_2026` | **81.3** | 81.2 | 70.8 | 69.7 | 65.6 | 34.2 |
| **Math** | **IMO-AnswerBench** | `Hwilner/imo-answerbench` | **75.2** | 74.2 | 69.8 | 68.5 | 57.2 | 26.9 |
| **General & Knowledge** | **IFEval** | `google/IFEval` | **93.6** | 93.0 | 91.5* | 89.8* | 94.8 | 45.3 |
| **General & Knowledge** | **IFBench** | `allenai/IFBench_test` | **75.8** | 75.0 | 64.5 | 59.2 | 73.5* | 44.0* |
| **General & Knowledge** | **GPQA Diamond** | `Idavidrein/gpqa` | **77.1** | 73.7 | 81.7* | 76.2* | 78.8* | 58.6* |
| **Agent** | **BFCL-V4** | `gorilla-llm/Berkeley-Function-Calling-Leaderboard` | **71.8** | 65.1 | 66.1* | 50.3* | 37.4 | 47.0* |
| **Agent** | **$\tau^2$-bench** | Sierra $\tau^2$-bench | **86.9** | 75.1 | 79.1* | 79.9* | 69.0* | 42.2* |
| **Code** | **SWE Verified** | `princeton-nlp/SWE-bench_Verified` | **62.8** | 41.6 | 53.1* | 38.8* | 44.2* | 14.0* |
| **Code** | **SciCode** | `SciCode1/SciCode` | **36.8** | 34.7 | 32.7* | 24.0 | 39.8 | 27.5 |

> **Official Evaluation Footnotes**:
> - All evaluations are conducted in thinking mode (`thinking_level="high"`). Recommended sampling parameters for `Spark-X2.6-4B`: `temperature=1.0`, `top_p=0.95`, `top_k=-1`.
> - `Gaokao 2026` consists of the five 2026 Chinese GAOKAO examinations (National I, National II, Beijing, Shanghai, Tianjin), each graded out of 150 points (`https://huggingface.co/datasets/XHToken/gaokao_math_2026`).

In [ ]:
# ==============================================================================
# Step 1: Sync `loop72-dlsa` Branch & Import Loop72 + Per-Visit Paged D-LSA Suite
# ==============================================================================
import importlib
import json
import os
import subprocess
import sys
import torch

REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-Recursion.git"
BRANCH = "loop72-dlsa"
REPO_DIR = "/content/Spark-X2.6-Recursion"

if os.path.exists("/content") and not os.path.exists( os.path.join(os.getcwd(), "sparkx_loop72_dlsa_benchmarks.py") ):
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)
elif os.path.exists(".git"):
    subprocess.run(["git", "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "checkout", BRANCH], check=False)
    subprocess.run(["git", "pull", "origin", BRANCH], check=False)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import sparkx_kv_codec
import sparkx_dlsa
import sparkx_paged_cache
import sparkx_sparse_attention
import sparkx_loop_v03_core as core
import load_inference_v03 as loader
import train_spark_dlsa
import sparkx_loop72_dlsa_benchmarks as bench

for _mod in (
    sparkx_kv_codec,
    sparkx_dlsa,
    sparkx_paged_cache,
    sparkx_sparse_attention,
    core,
    loader,
    train_spark_dlsa,
    bench,
):
    importlib.reload(_mod)

print(f"Working directory : {os.getcwd()}")
print(f"PyTorch version   : {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device        : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB)")

In [ ]:
# ==============================================================================
# Step 2: Load `Spark-X2.6-4B-Loop72-LSA` (`0` Engram Allocation) & Audit Capacity
# ==============================================================================
MODEL_ID = "XHToken/Spark-X2.6-4B"

if "model" in globals() and hasattr(model, "base"):
    print("[Info] Reconfiguring already-loaded model in memory for Loop72 + Per-Visit Paged D-LSA...")
    model = loader.configure_model_for_loop72_lsa(model)
else:
    model, tokenizer = loader.load_sparkx_loop72_lsa(model_id=MODEL_ID)

manifest = model.execution_manifest()
print("=== Spark-X2.6-4B-Loop72-LSA Execution Manifest ===")
print(json.dumps(manifest, indent=2))

assert manifest["engram_constructed"] is False, "Engram must NOT be constructed on the loop72-dlsa branch!"
assert manifest["total_virtual_layers"] == 72, f"Expected 72 total visits, got {manifest['total_virtual_layers']}"
assert manifest["global_visits"] == 18, f"Expected 18 global visits, got {manifest['global_visits']}"
assert manifest["local_visits"] == 54, f"Expected 54 local visits, got {manifest['local_visits']}"
assert manifest["dlsa_selector_parameters"] == 1_769_472

print("\n=== 72-Visit KV Capacity & Compression Table (32K / 128K / 1M / 5M Tokens) ===")
for n_tok in (32_768, 131_072, 1_000_000, 5_000_000):
    cap = sparkx_kv_codec.compute_loop72_capacity_table(n_tokens=n_tok)
    print(
        f"N={n_tok:>9,d} | Unbounded 72L BF16: {cap['unbounded_all72_bf16_gib']:>7.2f} GiB | "
        f"54 Local Rings: {cap['local_rings_mib']:.0f} MiB | "
        f"18 Global Visits (BF16: {cap['global_payload_gib']['16bit_gib']:.2f} GiB, "
        f"INT8: {cap['global_payload_gib']['8bit_gib']:.2f} GiB, "
        f"Mixed-Rotary-BF16: {cap['mixed_rotary_bf16_gib']:.2f} GiB, "
        f"INT4: {cap['global_payload_gib']['4bit_gib']:.2f} GiB)"
    )

print("\n=== Cold KV Page Codec Quality Audit (64-Token Page, 64 Rotated + 192 Free Dims) ===")
codec_summary = bench.evaluate_cold_kv_codecs_summary()
print(json.dumps(codec_summary, indent=2))

In [ ]:
# ==============================================================================
# Step 3: Pillar 1 — Modern 2025-2026 Held-Out Text Perplexity & Paged D-LSA Audit
#         Compares: 36L_Base_Dense vs 72L_Loop_Dense vs 72L_Loop_DLSA_Flat vs Tree
# ==============================================================================
SEQ_LEN_PPL = 4096  # Increase to 8192 / 16384 / 32768 on high-VRAM GPUs

PPL_REPORT = bench.evaluate_modern_text_perplexity_and_dlsa(
    model,
    tokenizer,
    seq_len=SEQ_LEN_PPL,
    calibrate_pca=True,
)

print("=== Modern 2025-2026 Held-Out Technical Text Perplexity & D-LSA Report ===")
print(json.dumps(PPL_REPORT, indent=2))

In [ ]:
# ==============================================================================
# Step 4: Pillar 2 — Per-Visit Sampled-Query KL + Focal-KL Selector Distillation
#         & 4-Way Positional/Sparse Evaluation (NativeRoPE vs YaRNRoPE x Dense/Sparse)
# ==============================================================================
DISTILL_SEQ_LEN = 2048
DISTILL_STEPS_PER_VISIT = 40

device = next(model.base.parameters()).device
modern_stream = bench.build_modern_long_eval_stream(target_tokens=DISTILL_SEQ_LEN + 256, tokenizer=tokenizer)
distill_ids = torch.tensor(
    [tokenizer.encode(modern_stream, add_special_tokens=False)[:DISTILL_SEQ_LEN]],
    dtype=torch.long,
    device=device,
)

print(f"Capturing per-visit Q/K/V across all 18 global visits (seq_len={distill_ids.shape[1]})...")
qkv_by_visit = train_spark_dlsa.capture_loop72_global_visit_qkv(model, distill_ids)
visit_samples = {gv: [(q_t, k_t)] for gv, (q_t, k_t, _) in qkv_by_visit.items()}

distill_cfg = train_spark_dlsa.VisitDistillConfig(
    steps=DISTILL_STEPS_PER_VISIT,
    sampled_queries=64,
    keys_per_step=min(1024, DISTILL_SEQ_LEN),
    lr=2e-3,
    init_with_pca=True,
)
DISTILL_REPORT = train_spark_dlsa.distill_loop72_visit_bank(
    model.dlsa_index_bank,
    visit_samples,
    cfg=distill_cfg,
)

print("=== Per-Visit Selector Distillation Summary (18 Global Visits) ===")
for gv_idx, rep in sorted(DISTILL_REPORT.items()):
    v_spec = sparkx_dlsa.LOOP72_GLOBAL_VISIT_SCHEDULE[gv_idx]
    print(
        f"GlobalVisit {gv_idx:>2d} (L{v_spec.physical_layer:>2d} {v_spec.stage:<7s} visit={v_spec.visit_index:>2d}) | "
        f"loss: {rep['initial_loss']:.4f} -> {rep['final_loss']:.4f} | "
        f"recall@K: {rep['initial_recall']:.3f} -> {rep['final_recall']:.3f}"
    )

print("\n=== 4-Way Positional & Sparse Ablation (`Dense/Sparse` x `NativeRoPE/YaRNRoPE`) ===")
FOUR_WAY_REPORT = train_spark_dlsa.run_loop72_four_way_eval(model, distill_ids[:, :1024])
print(json.dumps(FOUR_WAY_REPORT, indent=2))

In [ ]:
# ==============================================================================
# Step 5: Pillar 3 — Significant Official Benchmarks (`GraphWalks BFS`, `MRCR v2`,
#         `Gaokao 2026`, `AIME 2026`, `HMMT Feb 26`, `IMO-AnswerBench`,
#         `AA-LCRv1.1`, `LongBench v2`, `IFEval`, `IFBench`, `GPQA Diamond`,
#         `BFCL-V4`, `SciCode`) + Live Hugging Face Dataset Stream Preview
# ==============================================================================
from IPython.display import Markdown, display

hf_samples = bench.try_load_hf_benchmark_samples(max_per_dataset=1)
print(f"Streamed official Hugging Face datasets: {list(hf_samples.keys())}")

BENCH_REPORT = bench.evaluate_selected_benchmark_tasks(
    model,
    tokenizer,
    max_new_tokens=512,
    graphwalks_nodes=64,
    mrcr_rounds=16,
)

print("\n=== Live Loop72 + Paged D-LSA Benchmark Task Summary ===")
print(f"Tasks Evaluated : {BENCH_REPORT['num_tasks']}")
print(f"Pass Rate (%)   : {BENCH_REPORT['pass_rate_pct']}%")
print(f"Mean Score      : {BENCH_REPORT['mean_score']}")
for t in BENCH_REPORT["tasks"]:
    status = "PASS" if t["passed"] else "FAIL"
    print(
        f"  [{status}] {t['category']:<19s} | {t['benchmark']:<26s} | "
        f"Live={t['score']:>6.1f} (Official Spark X2.6-4B={t['official_spark_x26_score']}) | "
        f"{t['latency_sec']:.2f}s"
    )

md_table = bench.format_official_comparison_markdown(BENCH_REPORT)
display(Markdown("### Official Benchmark Comparison Table vs. Live `Loop72 + D-LSA`\n\n" + md_table))